In [0]:
%run "/Workspace/Users/collinlee-@live.ca/fake_general_ledger/2.1 silver_transformation_framework"

In [0]:
# ---------------------------------------------------------
# 0. Silver Transformation Framework Parameters
# ---------------------------------------------------------

schema = StructType([
    StructField("entryno", DecimalType(10,2), True),
    StructField("date", DateType(), True),
    StructField("territory_key", IntegerType(), True),
    StructField("account_key", IntegerType(), True),
    StructField("details", StringType(), True),
    StructField("amount", DecimalType(10,2), True),

])

bronze_table = "fake_general_ledger.default.bronze_general_ledger"

silver_table = "fake_general_ledger.default.silver_general_ledger"  

merge_condition = """target.entryno = source.entryno"""

required_columns = [
    "entryno",
    "date",
    "territory_key",
    "account_key",
    "details",
    "amount",
]

primary_key_columns = [
    "entryno",
]

order_column = "entryno"



In [0]:
# ---------------------------------------------------------
# 1. Silver Transformation Driver Code
# ---------------------------------------------------------

bronze_df = spark.table(
    bronze_table
)

bronze_df = standardize_column_names(
    bronze_df
)

silver_df = apply_silver_schema(
    bronze_df,
    schema
)

silver_df, invalid_df = validate_required_columns(
    silver_df,
    required_columns
)

silver_df = deduplicate_records(
    silver_df,
    primary_key_columns,
    order_column
)

silver_df = apply_silver_metadata(
    silver_df
)

silver_df = rename_bronze_metadata(
    silver_df
)

silver_df = drop_excess_bronze_metadata(
    silver_df
)

merge_to_silver(
    spark,
    silver_df,
    silver_table,
    merge_condition
)